# Paying per call (x402 and MPP)

Free trial data needs nothing. For anything outside the trial, an agent without an account
pays each call in USDC, about $0.02: x402 on Base, MPP on Tempo or MPP on Solana.
Payments are **off by default**. The client pays only with a wallet **and** `max_usd_per_call`,
and refuses anything more expensive before signing.

Use a dedicated wallet with a small balance. Never commit the private key.

In [1]:
%pip install -q "econdash[x402,solana]"   # add [tempo] for MPP on Tempo (Python 3.11+)

In [2]:
import os

from eth_account import Account
from solders.keypair import Keypair  # noqa: F401  (for the Solana line)

from econdash import EconDash

ed = EconDash(
    x402_signer=Account.from_key(os.environ["AGENT_PRIVATE_KEY"]),  # USDC on Base
    # solana_keypair=Keypair.from_base58_string(os.environ["AGENT_SOLANA_KEY"]),
    max_usd_per_call=0.05,
)
print("rails:", ed.payment_methods)

# Inside the free trial: no payment
print(ed.get_series("gdp-nominal", "USA", start=2023).data[-1])

# Zimbabwe is outside the free trial, so this call is paid once
s = ed.get_series("gdp-nominal", "ZWE", start=2015)
print(s.indicator_name_en, [(p.date.year, round(p.value / 1e9, 1)) for p in s.data])

rails: ['x402']
date=datetime.date(2025, 1, 1) value=30769700000000.0 country='USA' country_name='United States' is_projection=None


GDP nominal [(2015, 20.0), (2016, 20.6), (2017, 51.0), (2018, 34.1), (2019, 33.4), (2020, 32.0), (2021, 41.3), (2022, 40.8), (2023, 35.9), (2024, 41.5), (2025, 51.2)]


In [3]:
from econdash import EconDashPaymentRequired

# Without a wallet the 402 comes back as an exception and nothing is paid
try:
    EconDash().get_series("gdp-nominal", "ZWE")
except EconDashPaymentRequired as e:
    accept = e.challenge["accepts"][0]
    print("x402:", int(accept["amount"]) / 1e6, "USDC on", accept["network"])
    for line in e.www_authenticate:
        print("MPP:", line.split('method="')[1].split('"')[0])

x402: 0.02 USDC on eip155:8453
MPP: tempo
MPP: solana
